# MercadoTico 360: Consultas y demostración (CouchDB)

Ejecute las celdas en orden. El notebook presenta el diseño de los documentos, las consultas principales, los índices, las agregaciones y la evolución del esquema implementado en CouchDB.

**NOTA:** Requiere haber cargado antes los datos con `Simulacion.ipynb`.

| Funcionalidad | Sección | Evidencia |
|---|---|---|
| Diseño documental y atributos por categoría | Documentos | Visualización de documentos y estructura |
| Índices para patrones de consulta | Índices | Comparación y utilidad de los índices |
| Búsqueda por categoría, precio y atributo | Búsquedas | Consultas antes y después de crear índices |
| Consulta y actualización de pedidos | Pedidos | Pedido anidado y cambio de estado |
| Agregaciones | MapReduce | Ventas por categoría, ticket promedio y productos más vendidos |
| Evolución del esquema | Evolución | Incorporación de una nueva categoría o atributo |

In [1]:
import json
import os
from getpass import getpass

import requests

try:                                   # carga el .env si existe
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass


def leer_config(nombre, pregunta, defecto=None, secreto=False):
    # Toma el valor de una variable de entorno; si no existe, lo pregunta.
    valor = os.getenv(nombre)
    if valor:
        return valor.strip()
    if secreto:
        return getpass(f"{pregunta}: ").strip()
    sufijo = f" [{defecto}]" if defecto else ""
    return input(f"{pregunta}{sufijo}: ").strip() or defecto


COUCHDB_HOST  = leer_config("COUCHDB_HOST", "Host/IP de CouchDB", "localhost")
COUCHDB_PORT  = leer_config("COUCHDB_PORT", "Puerto de CouchDB")
COUCHDB_USER  = leer_config("COUCHDB_USER", "Usuario de CouchDB", "admin")
COUCHDB_PASS  = leer_config("COUCHDB_PASS", "Contraseña de CouchDB", secreto=True)
DATABASE_NAME = leer_config("DATABASE_NAME", "Nombre de la base de datos", "mercadotico360")

DB_URL = f"http://{COUCHDB_HOST}:{COUCHDB_PORT}/{DATABASE_NAME}"

SESSION = requests.Session()
SESSION.auth = (COUCHDB_USER, COUCHDB_PASS)

## Funciones de apoyo
Todas las consultas usan `buscar()`, que imprime siempre las mismas métricas (resultados, documentos examinados y tiempo) para poder comparar.

In [ ]:
RESULTADOS = {}      # guarda los resultados de las pruebas


def registrar(prueba, ok, detalle=""):
    RESULTADOS[prueba] = (ok, detalle)
    estado = "Correcto" if ok else "Revisar"
    if detalle:
        print(f"{estado}: {prueba} - {detalle}")
    else:
        print(f"{estado}: {prueba}")
    

def buscar(selector, use_index=None, limit=100000, titulo=""):
    # ejecuta una consulta Mango (_find) e imprime sus métricas de rendimiento.
    payload = {"selector": selector, "limit": limit, "execution_stats": True}
    if use_index:
        payload["use_index"] = [use_index, use_index]
        payload["allow_fallback"] = False
    res = SESSION.post(f"{DB_URL}/_find", json=payload, timeout=300)
    res.raise_for_status()
    data = res.json()
    stats = data.get("execution_stats", {})
    print(f"{titulo}: {len(data.get('docs', []))} resultados | "
          f"docs examinados: {stats.get('total_docs_examined')} | "
          f"{stats.get('execution_time_ms')} ms")
    if data.get("warning"):
        print("   aviso de CouchDB:", data["warning"].split(".")[0])
    return data


def guardar_design_doc(ddoc):
    # crea o actualiza un design document (vistas, update handlers, validaciones).
    url = f"{DB_URL}/{ddoc['_id']}"
    actual = SESSION.get(url, timeout=10)
    if actual.status_code == 200:
        ddoc["_rev"] = actual.json()["_rev"]      # necesario para actualizar
    SESSION.put(url, json=ddoc, timeout=60).raise_for_status()


def verificar_datos():
    # comprueba que existan productos, clientes y pedidos cargados. 
    for tipo in ("product", "customer", "order"):
        r = SESSION.post(f"{DB_URL}/_find", timeout=120,
                         json={"selector": {"type": tipo}, "limit": 1, "fields": ["_id"]}).json()
        if not r.get("docs"):
            raise RuntimeError(f"No hay documentos '{tipo}'. Ejecute primero Simulacion.ipynb.")
    print("La base de datos contiene productos, clientes y pedidos.")


verificar_datos()

RuntimeError: No hay documentos 'product'. Ejecute primero Simulacion.ipynb.

## Documentos y atributos variables
**Qué se embebe y qué se referencia**
- `producto`: documento propio; sus `atributos` van **embebidos** y cambian según la categoría.
- `pedido`: las `lineas_detalle` van **embebidas** (con *snapshot* de nombre, categoría y precio); `cliente_id` y `producto_id` son **referencias**.
- `cliente`: documento propio, para que sus datos cambien sin tocar los pedidos.

In [ ]:
def mostrar_productos_variables(categorias=("Ropa", "Tecnologia", "Artesanias")):
    # muestra un producto de cada categoría: los atributos son distintos en cada una.
    for cat in categorias:
        r = SESSION.post(f"{DB_URL}/_find", timeout=60,
                         json={"selector": {"type": "product", "categoria": cat}, "limit": 1}).json()
        if r.get("docs"):
            p = r["docs"][0]
            print(f"{p['_id']} | {p['categoria']} | atributos: {json.dumps(p['atributos'], ensure_ascii=False)}")


def mostrar_pedido_anidado(pedido_id="PED-000001"):
    # muestra un pedido completo con sus líneas de detalle anidadas.
    pedido = SESSION.get(f"{DB_URL}/{pedido_id}", timeout=10).json()
    print(json.dumps(pedido, indent=2, ensure_ascii=False))
    return pedido


print("--- Productos con atributos distintos por categoría ---")
mostrar_productos_variables()
print("\n--- Pedido con estructura anidada ---")
pedido_demo = mostrar_pedido_anidado()
registrar("Diseño y atributos variables", "lineas_detalle" in pedido_demo)

## Índices para patrones de consulta frecuentes

| Índice | Campos | Patrón de consulta que acelera |
|---|---|---|
| `idx_product_category` | `type`, `categoria` | Listar productos de una categoría (navegación del catálogo) |
| `idx_product_price` | `type`, `precio` | Buscar productos dentro de un rango de precios |
| `idx_technology_ram` | `type`, `categoria`, `atributos.ram_gb` | Filtrar por un atributo propio de una categoría (RAM en Tecnologia) |
| `idx_pedidos_cliente` | `type`, `cliente_id` | Recuperar el historial de pedidos de un cliente |

Mango selecciona un índice para ejecutar la consulta; las condiciones que no puedan resolverse directamente con ese índice se evalúan posteriormente sobre los documentos candidatos.

In [ ]:
INDICES = {
    "idx_product_category": ["type", "categoria"],
    "idx_product_price": ["type", "precio"],
    "idx_technology_ram": ["type", "categoria", "atributos.ram_gb"],
    "idx_pedidos_cliente": ["type", "cliente_id"],
}


def crear_indice(nombre):
    payload = {"index": {"fields": INDICES[nombre]}, "ddoc": nombre, "name": nombre, "type": "json"}
    respuesta = SESSION.post(f"{DB_URL}/_index", json=payload, timeout=300)
    respuesta.raise_for_status()
    res = respuesta.json()
    print(f"  {nombre}: {res.get('result')}")          # 'created' o 'exists'


def crear_indices():
    for nombre in INDICES:
        crear_indice(nombre)


def eliminar_indice(nombre):
    # elimina un índice por su nombre para comparar el rendimiento.
    for idx in SESSION.get(f"{DB_URL}/_index", timeout=30).json().get("indexes", []):
        if idx["name"] == nombre and idx["type"] == "json":
            SESSION.delete(f"{DB_URL}/_index/{idx['ddoc']}/json/{nombre}", timeout=60)
            print(f"  {nombre}: eliminado")


print("Creando índices (puede tardar unos segundos)...")
crear_indices()
registrar("Índices creados", True, f"{len(INDICES)} índices")

## Búsquedas por categoría, rango de precio y atributo específico
Se ejecuta la misma consulta antes y después de crear los índices para comparar los documentos examinados y el tiempo de ejecución.

In [ ]:
CONSULTA_CATEGORIA_PRECIO = {
    "type": "product",
    "categoria": "Tecnologia",
    "precio": {"$gte": 100000, "$lte": 500000},
}


def comparar_antes_despues():
    # mide la consulta categoría + precio sin índices y con índice.
    print("ANTES: se eliminan los índices de productos")
    for nombre in ("idx_product_category", "idx_product_price", "idx_technology_ram"):
        eliminar_indice(nombre)
    antes = buscar(CONSULTA_CATEGORIA_PRECIO, titulo="SIN índices")

    print("\nDESPUÉS: se vuelven a crear")
    crear_indices()
    despues = buscar(CONSULTA_CATEGORIA_PRECIO, use_index="idx_product_price", titulo="CON idx_product_price")

    ea, ed = antes.get("execution_stats", {}), despues.get("execution_stats", {})
    mejora = ea.get("total_docs_examined", 0) / max(ed.get("total_docs_examined", 1), 1)
    print(f"\nDocumentos examinados: {ea.get('total_docs_examined')} -> {ed.get('total_docs_examined')} "
          f"({mejora:.1f}x menos)")
    ok = len(antes.get("docs", [])) == len(despues.get("docs", [])) and mejora > 1
    registrar("Comparación antes/después de índices", ok, f"{mejora:.1f}x menos documentos examinados")


def buscar_categoria_precio_atributo(ram_gb=16, precio_min=100000, precio_max=1500000):
    # tecnologia + rango de precio + atributo específico (RAM).
    selector = {
    "type": "product",
    "categoria": "Tecnologia",
    "atributos.ram_gb": ram_gb,
    "precio": {"$gte": precio_min, "$lte": precio_max},
    }
    data = buscar(selector, use_index="idx_technology_ram", titulo=f"Tecnologia, {ram_gb} GB RAM, ₡{precio_min}-₡{precio_max}")
    if data.get("docs"):
        ejemplo = data["docs"][0]
        print(f"   ejemplo: {ejemplo['nombre']} | ₡{ejemplo['precio']} | {ejemplo['atributos']}")
    registrar("Búsqueda por categoría, precio y atributo", bool(data.get("docs")), f"{len(data.get('docs', []))} productos")


comparar_antes_despues()
print()
buscar_categoria_precio_atributo()

## Pedidos de un cliente y cambio de estado
Para actualizar solo el estado se usa un *update handler* de CouchDB: el cliente envía únicamente `{"estado": ...}` y el servidor aplica el cambio, sin que el código tenga que reenviar el pedido completo.
(CouchDB internamente guarda una nueva revisión del documento.)

In [ ]:
def instalar_update_handler():
    guardar_design_doc({
        "_id": "_design/pedidos",
        "updates": {
            "cambiar_estado": (
                "function (doc, req) {"
                "  if (!doc) { return [null, 'pedido no existe']; }"
                "  doc.estado = JSON.parse(req.body).estado;"
                "  return [doc, 'ok'];"
                "}"
            )
        },
    })


def cambiar_estado_pedido(pedido_id, nuevo_estado):
    url = f"{DB_URL}/_design/pedidos/_update/cambiar_estado/{pedido_id}"
    return SESSION.put(url, json={"estado": nuevo_estado}, timeout=30)


def recuperar_y_actualizar_pedido(cliente_id="CLI-000001", nuevo_estado="completado"):
    # 1. Recuperar los pedidos completos del cliente
    data = buscar(
    {"type": "order", "cliente_id": cliente_id},
    use_index="idx_pedidos_cliente",
    limit=1000,
    titulo=f"Pedidos de {cliente_id}")
    pedidos = data.get("docs", [])
    if not pedidos:
        registrar("Pedidos y actualización", False, "el cliente no tiene pedidos")
        return

    # 2. Actualizar el estado de uno (solo se envía el campo 'estado')
    pedido = next((p for p in pedidos if p["estado"] != nuevo_estado), pedidos[0])   # uno que sí cambie
    anterior = pedido["estado"]
    instalar_update_handler()
    res = cambiar_estado_pedido(pedido["_id"], nuevo_estado)

    # 3. Comprobar el cambio
    actualizado = SESSION.get(f"{DB_URL}/{pedido['_id']}", timeout=10).json()
    print(f"\n{pedido['_id']}: '{anterior}' -> '{actualizado['estado']}'  (HTTP {res.status_code})")
    lineas_iguales = pedido["lineas_detalle"] == actualizado["lineas_detalle"]
    print(
    f"Líneas de detalle conservadas: "
    f"{len(actualizado['lineas_detalle'])} | "
    f"sin cambios: {lineas_iguales}")   
    registrar("Pedidos y actualización de estado", actualizado["estado"] == nuevo_estado and lineas_iguales,
              f"{len(pedidos)} pedidos recuperados")


recuperar_y_actualizar_pedido("CLI-000001")

## Agregaciones con MapReduce
Tres vistas sobre los pedidos (se excluyen los cancelados en ventas y productos vendidos):

| Vista | Qué responde | Reduce |
|---|---|---|
| `ventas_por_categoria` | Ventas totales por categoría (usa el *snapshot* de cada línea) | `_sum` |
| `ticket_por_estado` | Cantidad, suma, mínimo y máximo de pedidos → ticket promedio | `_stats` |
| `productos_mas_vendidos` | Unidades vendidas por producto | `_sum` |

La primera consulta construye la vista y puede tardar un poco.

In [ ]:
def crear_vistas_agregacion():
    guardar_design_doc({
        "_id": "_design/agregaciones",
        "views": {
            "ventas_por_categoria": {
                "map": (
                    "function (doc) { if (doc.type === 'order' && doc.estado !== 'cancelado') {"
                    " doc.lineas_detalle.forEach(function (l) { emit(l.categoria_snapshot, l.subtotal); }); } }"
                ),
                "reduce": "_sum",
            },
            "ticket_por_estado": {
                "map": "function (doc) { if (doc.type === 'order') { emit(doc.estado, doc.monto_total); } }",
                "reduce": "_stats",
            },
            "productos_mas_vendidos": {
                "map": (
                    "function (doc) { if (doc.type === 'order' && doc.estado !== 'cancelado') {"
                    " doc.lineas_detalle.forEach(function (l) { emit(l.producto_id, l.cantidad); }); } }"
                ),
                "reduce": "_sum",
            },
        },
    })


def consultar_vista(nombre):
    url = f"{DB_URL}/_design/agregaciones/_view/{nombre}?group=true"
    respuesta = SESSION.get(url, timeout=600)
    respuesta.raise_for_status()
    return respuesta.json().get("rows", [])


def ejecutar_agregaciones():
    crear_vistas_agregacion()

    print("--- Ventas por categoría ---")
    ventas = sorted(consultar_vista("ventas_por_categoria"), key=lambda r: r["value"], reverse=True)
    for r in ventas:
        print(f"  {str(r['key']):<18} ₡{r['value']:>14,.2f}")

    print("\n--- Ticket promedio por estado de pedido ---")
    for r in consultar_vista("ticket_por_estado"):
        v = r["value"]
        print(f"  {r['key']:<12} pedidos: {v['count']:>6} | promedio: ₡{v['sum'] / v['count']:>9,.2f} "
              f"| mín: ₡{v['min']:,.2f} | máx: ₡{v['max']:,.2f}")

    print("\n--- Top 5 productos más vendidos (unidades) ---")
    top = sorted(consultar_vista("productos_mas_vendidos"), key=lambda r: r["value"], reverse=True)[:5]
    for r in top:
        print(f"  {r['key']}  {r['value']} unidades")

    registrar("Agregaciones MapReduce", bool(ventas and top), "3 vistas consultadas")


ejecutar_agregaciones()

## Evolución del esquema sin migración
Se demuestra con tres acciones sobre la misma base de datos, sin `ALTER TABLE` ni migración:
1. Una **categoría nueva** con atributos nunca vistos.
2. Un **atributo nuevo** en un producto existente.
3. Una **validación** (`validate_doc_update`) que mantiene la consistencia: rechaza documentos inválidos aunque el esquema sea flexible.

In [ ]:
def instalar_validacion():
    guardar_design_doc({
        "_id": "_design/validacion",
        "validate_doc_update": (
            "function (newDoc, oldDoc, userCtx) {"
            "  if (newDoc._deleted || newDoc._id.indexOf('_design/') === 0) { return; }"
            "  if (['product', 'customer', 'order'].indexOf(newDoc.type) === -1) {"
            "    throw ({forbidden: 'tipo de documento inválido'});"
            "  }"
            "  if (newDoc.type === 'product') {"
            "    if (!newDoc.categoria) { throw ({forbidden: 'categoría requerida'}); }"
            "    if (typeof newDoc.precio !== 'number' || newDoc.precio < 0) {"
            "      throw ({forbidden: 'precio inválido'});"
            "    }"
            "  }"
            "  if (newDoc.type === 'order' && "
            "      (!newDoc.lineas_detalle || newDoc.lineas_detalle.length === 0)) {"
            "    throw ({forbidden: 'el pedido necesita líneas'});"
            "  }"
            "}"
        )
    })


def demostrar_evolucion_esquema():
    instalar_validacion()
    print("[1] Validación instalada (consistencia)\n")

    # A) Categoría completamente nueva, con sus propios atributos
    nuevo = {
        "_id": "PROD-999999", "type": "product",
        "nombre": "Drone de Monitoreo Agrícola", "categoria": "Tecnologia_Agricola",
        "precio": 450000, "stock": 10,
        "atributos": {"autonomia_minutos": 45, "camara_multiespectral": True, "alcance_km": 12.5},
    }
    r = SESSION.put(f"{DB_URL}/PROD-999999", json=nuevo, timeout=30)
    estado = {201: "creada", 202: "creada", 409: "ya existía"}.get(r.status_code, f"error {r.status_code}")
    print(f"[2] Categoría nueva 'Tecnologia_Agricola': {estado}")
    hallados = buscar({"type": "product", "categoria": "Tecnologia_Agricola"},
                      use_index="idx_product_category", titulo="    consulta de la categoría nueva")

    # B) Atributo nuevo en un producto existente
    ropa = SESSION.post(f"{DB_URL}/_find", timeout=60,
                        json={"selector": {"type": "product", "categoria": "Ropa"}, "limit": 1}).json()["docs"][0]
    ropa["atributos"]["certificacion_ecologica"] = True
    r2 = SESSION.put(f"{DB_URL}/{ropa['_id']}", json=ropa, timeout=30)
    print(f"[3] Atributo nuevo 'certificacion_ecologica' en {ropa['_id']}: HTTP {r2.status_code}")

    # C) Consistencia: un producto inválido es rechazado por la base de datos
    invalido = {"type": "product", "categoria": "Ropa", "nombre": "Prueba", "precio": -5}
    r3 = SESSION.post(DB_URL, json=invalido, timeout=30)
    print(f"[4] Producto con precio negativo: HTTP {r3.status_code} -> {r3.json().get('reason')}")

    registrar("Evolución del esquema",
              r.status_code in (201, 202, 409) and bool(hallados.get("docs")) and r3.status_code == 403 and r2.status_code in (201, 202),
              "categoría nueva, atributo nuevo y validación")


demostrar_evolucion_esquema()

## Resumen de pruebas

In [ ]:
print("RESUMEN DE PRUEBAS - MERCADOTICO 360")
for prueba, (ok, detalle) in RESULTADOS.items():
    estado = "Correcto" if ok else "Revisar"
    print(f"{estado}: {prueba} - {detalle}")
print(
    f"\nPruebas correctas: "
    f"{sum(ok for ok, _ in RESULTADOS.values())}/{len(RESULTADOS)}"
)